# 4. Images as Data

**AI and Economics Summer School** — Andrea Ciccarone

Everything in notebook 2 carries over: we embed, then do statistics on the vectors.
The only thing that changes is the encoder.

What is different and genuinely useful about images is **CLIP**: images and text
land in the *same* vector space, so you can

- classify images with no training data at all (zero-shot), and
- compare an image to a sentence on a single scale.

That second property is what makes the political-ads application in the lecture
possible. Runs on CPU; a GPU runtime in Colab makes it faster but is not required.

In [ ]:
!pip -q install torch torchvision open_clip_torch

In [ ]:
import numpy as np
import torch, open_clip
import matplotlib.pyplot as plt
from torchvision.datasets import CIFAR10
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import adjusted_rand_score, accuracy_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
plt.rcParams["figure.dpi"] = 120

## 1. A small image corpus

CIFAR-10: ten visually distinct classes. Small enough to download in seconds and
run on CPU. Substantively boring, but the workflow is identical to the one you
would run on news frames or political ads.

In [ ]:
ds = CIFAR10(root="./data", train=False, download=True)
classes = ds.classes
print(classes)

N = 1000
idx = np.random.default_rng(0).choice(len(ds), N, replace=False)
images = [ds[i][0] for i in idx]
labels = np.array([ds[i][1] for i in idx])

fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for ax, i in zip(axes.ravel(), range(16)):
    ax.imshow(images[i]); ax.axis("off")
    ax.set_title(classes[labels[i]], fontsize=7)
plt.tight_layout(); plt.show()

## 2. Embed the images

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k")
tokenizer = open_clip.get_tokenizer("ViT-B-32")
model = model.to(device).eval()

batch = torch.stack([preprocess(im) for im in images]).to(device)
with torch.no_grad():
    feats = []
    for i in range(0, len(batch), 128):
        f = model.encode_image(batch[i:i+128])
        feats.append(f / f.norm(dim=-1, keepdim=True))   # normalise -> cosine
    E_img = torch.cat(feats).cpu().numpy()

print("image embeddings:", E_img.shape)

## 3. Zero-shot classification

Embed a *sentence* for each class, then assign each image to the nearest sentence.
No training data, no labels, no fitting.

In [ ]:
prompts = [f"a photo of a {c}" for c in classes]
with torch.no_grad():
    t = model.encode_text(tokenizer(prompts).to(device))
    E_txt = (t / t.norm(dim=-1, keepdim=True)).cpu().numpy()

sim = E_img @ E_txt.T              # cosine similarity, images x classes
pred = sim.argmax(1)
print("zero-shot accuracy:", round(accuracy_score(labels, pred), 3))
print("(chance would be 0.100)")

This is the single most useful thing in the notebook for applied work. You can
score any image against any concept you can put into words, with no labeled data.

The prompt is a research choice. Try changing it.

In [ ]:
for template in ["{}", "a photo of a {}", "a blurry photo of a {}", "a {} in the wild"]:
    p = [template.format(c) for c in classes]
    with torch.no_grad():
        tt = model.encode_text(tokenizer(p).to(device))
        Et = (tt / tt.norm(dim=-1, keepdim=True)).cpu().numpy()
    acc = accuracy_score(labels, (E_img @ Et.T).argmax(1))
    print(f"{template:28s} accuracy {acc:.3f}")

> **Report your prompt.** Accuracy moves with wording, which means the prompt is a
> researcher degree of freedom exactly like a preprocessing choice. Fix it in
> advance, write it down, and show that the result survives reasonable alternatives.

## 4. Embeddings as features for a supervised model

If you *do* have a few hundred labels, fitting a simple classifier on frozen
embeddings usually beats zero-shot and costs almost nothing. This is transfer
learning in its cheapest form, and it is what the ads partisanship model does.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(E_img, labels, test_size=0.5,
                                      stratify=labels, random_state=0)
lr = LogisticRegression(max_iter=3000).fit(Xtr, ytr)
print("linear probe on 500 labels:", round(lr.score(Xte, yte), 3))
print("zero-shot on the same test set:", round(accuracy_score(yte, (Xte @ E_txt.T).argmax(1)), 3))

## 5. Unsupervised: cluster the images

Same code as notebook 2. The encoder changed; nothing else did.

In [ ]:
km = KMeans(n_clusters=10, n_init=10, random_state=0).fit(E_img)
print("ARI vs true classes:", round(adjusted_rand_score(labels, km.labels_), 3))

Z = PCA(n_components=2, random_state=0).fit_transform(E_img)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].scatter(Z[:, 0], Z[:, 1], c=labels, cmap="tab10", s=8)
axes[0].set_title("coloured by true class")
axes[1].scatter(Z[:, 0], Z[:, 1], c=km.labels_, cmap="tab10", s=8)
axes[1].set_title("coloured by k-means cluster")
for a in axes: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
# Read the clusters: show the images closest to each centroid.
fig, axes = plt.subplots(10, 6, figsize=(8, 13))
for c in range(10):
    m = np.where(km.labels_ == c)[0]
    order = m[np.argsort(-(E_img[m] @ km.cluster_centers_[c]))][:6]
    for ax, i in zip(axes[c], order):
        ax.imshow(images[i]); ax.axis("off")
    axes[c][0].set_ylabel(f"c{c}", rotation=0)
    axes[c][0].axis("on"); axes[c][0].set_xticks([]); axes[c][0].set_yticks([])
plt.tight_layout(); plt.show()

This is the image equivalent of reading the documents nearest each centroid. Do it
every time. It is the only way to know whether a visual cluster is a theme or an
artefact of lighting, resolution or watermarks.

## 6. Cross-modal: naming clusters with words

Because text and images share the space, we can label a discovered cluster by
asking which sentence its centroid is closest to. No human coding required for a
first pass.

In [ ]:
concepts = ["an animal", "a vehicle", "something that flies", "something in water",
            "a pet", "a wild animal", "a machine", "a bird"]
with torch.no_grad():
    tc = model.encode_text(tokenizer(concepts).to(device))
    E_con = (tc / tc.norm(dim=-1, keepdim=True)).cpu().numpy()

for c in range(10):
    centroid = km.cluster_centers_[c]
    centroid = centroid / np.linalg.norm(centroid)
    s = E_con @ centroid
    best = np.argsort(-s)[:2]
    true_mode = classes[np.bincount(labels[km.labels_ == c]).argmax()]
    print(f"cluster {c}: '{concepts[best[0]]}', '{concepts[best[1]]}'"
          f"   [most common true class: {true_mode}]")

## Exercises

1. Build a two-class problem (say `cat` vs `dog`) and compare zero-shot AUC to a
   linear probe trained on 50, 100 and 500 labels. Where is the crossover?
2. Write three different prompt sets for the same concept and report the spread in
   accuracy. That spread is your measurement uncertainty.
3. Cluster with `K = 5` and `K = 20`. At which K can you still name every cluster
   from its nearest images?
4. Take the cross-modal idea further: define two opposing sentences (e.g.
   "a threatening scene" vs "a reassuring scene"), project every image onto the
   difference between their embeddings, and use that projection as a continuous
   variable. This is the basic move behind concept-based measures of image content.